In [1]:
!pip install ultralytics -q
import json
import os
import shutil
import cv2
from pathlib import Path
from ultralytics import YOLO

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 22.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.0/91.0 kB 3.6 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [3]:
# Define paths based on Kaggle input directory
json_path = '/kaggle/input/datasets/truthisneverlinear/dentex-challenge-2023/training_data/training_data/quadrant/train_quadrant.json' 
img_dir = '/kaggle/input/datasets/truthisneverlinear/dentex-challenge-2023/training_data/training_data/quadrant/xrays/'
output_dir = '/kaggle/working/yolo_quadrants/'

os.makedirs(output_dir + 'images/train', exist_ok=True)
os.makedirs(output_dir + 'labels/train', exist_ok=True)

with open(json_path, 'r') as f:
    coco_data = json.load(f)

# Map image IDs to filenames and dimensions
images = {img['id']: img for img in coco_data['images']}

for ann in coco_data['annotations']:
    img_info = images[ann['image_id']]
    img_width, img_height = img_info['width'], img_info['height']
    
    # COCO bbox: [x_min, y_min, width, height]
    x_min, y_min, w, h = ann['bbox']
    
    # Convert to YOLO: normalized x_center, y_center, width, height
    x_center = (x_min + w / 2) / img_width
    y_center = (y_min + h / 2) / img_height
    w_norm = w / img_width
    h_norm = h / img_height
    
    # Quadrant classes in DENTEX are typically 0-3 (Top-Right, Top-Left, Bottom-Left, Bottom-Right)
    class_id = ann['category_id'] 
    
    # Write to YOLO label file
    label_path = os.path.join(output_dir, 'labels/train', img_info['file_name'].replace('.jpg', '.txt').replace('.png', '.txt'))
    with open(label_path, 'a') as label_file:
        label_file.write(f"{class_id} {x_center} {y_center} {w_norm} {h_norm}\n")

# Copy images to working directory (Kaggle requires writable paths for YOLO training)
for img in coco_data['images']:
    src = os.path.join(img_dir, img['file_name'])
    dst = os.path.join(output_dir, 'images/train', img['file_name'])
    if os.path.exists(src):
        shutil.copy(src, dst)

In [4]:
yaml_content = f"""
train: {output_dir}images/train
val: {output_dir}images/train  # Split this properly later using sklearn.model_selection

nc: 4
names: ['Quadrant_1', 'Quadrant_2', 'Quadrant_3', 'Quadrant_4']
"""

with open('/kaggle/working/quadrant_data.yaml', 'w') as f:
    f.write(yaml_content)

In [5]:
# Initialize a lightweight YOLOv8 model for the coordinator
model = YOLO('yolov8n.pt') 

# Train using both GPUs (0 and 1)
results = model.train(
    data='/kaggle/working/quadrant_data.yaml',
    epochs=50,
    imgsz=640,
    batch=32, # Increase batch size since you have 2 GPUs (16x2)
    device=[0, 1], 
    project='/kaggle/working/runs',
    name='coordinator_quadrant'
)

Ultralytics 8.4.164 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/quadrant_data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mo

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/50       2.1G      1.247      1.543      1.211        102        640: 100% ━━━━━━━━━━━━ 22/22 1.2it/s 17.7s0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 11/11 1.8s/it 20.3s0.3s
                   all        693       2772      0.497      0.963      0.567      0.372

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/50      2.11G      1.146      1.352      1.152        104        640: 100% ━━━━━━━━━━━━ 22/22 1.2it/s 18.0s0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 11/11 1.9s/it 21.3s0.4s
                   all        693       2772      0.485      0.788      0.555      0.346

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/50      2.11G      1.123      1.341      1.144         87        640: 

In [8]:
import json
import os
import cv2
from ultralytics import YOLO

# 1. Load your highly accurate coordinator model
coordinator = YOLO('/kaggle/working/runs/coordinator_quadrant/weights/best.pt')

# 2. Define paths for the enumeration dataset
# 2. Define paths for the enumeration dataset
json_path = '/kaggle/input/datasets/truthisneverlinear/dentex-challenge-2023/training_data/training_data/quadrant_enumeration/train_quadrant_enumeration.json'
img_dir = '/kaggle/input/datasets/truthisneverlinear/dentex-challenge-2023/training_data/training_data/quadrant_enumeration/xrays/'
output_base = '/kaggle/working/enumerators_data/'

# Create subdirectories for each of the 4 models
for q in range(1, 5):
    os.makedirs(f'{output_base}Q{q}/images/train', exist_ok=True)
    os.makedirs(f'{output_base}Q{q}/labels/train', exist_ok=True)

# 3. Load COCO JSON for the teeth enumeration
with open(json_path, 'r') as f:
    coco_data = json.load(f)

# Map annotations to their respective image IDs
img_annotations = {}
for ann in coco_data['annotations']:
    img_id = ann['image_id']
    if img_id not in img_annotations:
        img_annotations[img_id] = []
    img_annotations[img_id].append(ann)

# 4. Process each image, crop, and shift bounding boxes
for img_info in coco_data['images']:
    img_id = img_info['id']
    img_name = img_info['file_name']
    img_path = os.path.join(img_dir, img_name)
    
    if not os.path.exists(img_path): continue
        
    img = cv2.imread(img_path)
    if img is None: continue
        
    # Use your coordinator to find the 4 quadrants
    results = coordinator(img, verbose=False)
    boxes = results[0].boxes
    
    for box in boxes:
        # Get quadrant class (0=Q1, 1=Q2, 2=Q3, 3=Q4)
        q_class = int(box.cls[0].item()) 
        q_num = q_class + 1 
        
        # Get absolute pixel coordinates of the detected quadrant
        qx1, qy1, qx2, qy2 = map(int, box.xyxy[0].tolist())
        
        # Crop the image
        q_crop = img[qy1:qy2, qx1:qx2]
        crop_h, crop_w = q_crop.shape[:2]
        
        if crop_h == 0 or crop_w == 0: continue
        
        # Save cropped image
        crop_img_name = f"{img_name.split('.')[0]}_Q{q_num}.jpg"
        cv2.imwrite(f'{output_base}Q{q_num}/images/train/{crop_img_name}', q_crop)
        
        # Write the shifted labels
        label_path = f'{output_base}Q{q_num}/labels/train/{crop_img_name.replace(".jpg", ".txt")}'
        with open(label_path, 'w') as f_label:
            if img_id in img_annotations:
                for ann in img_annotations[img_id]:
                    # Original full-image tooth bbox
                    tx, ty, tw, th = ann['bbox']
                    tx_center = tx + tw/2
                    ty_center = ty + th/2
                    
                    # Check if the center of the tooth falls inside this specific quadrant crop
                    if qx1 <= tx_center <= qx2 and qy1 <= ty_center <= qy2:
                        
                        # SHIFT COORDINATES: Subtract quadrant's top-left corner
                        new_x_center = tx_center - qx1
                        new_y_center = ty_center - qy1
                        
                        # Normalize relative to the new crop dimensions for YOLO
                        norm_x = max(0, min(1, new_x_center / crop_w))
                        norm_y = max(0, min(1, new_y_center / crop_h))
                        norm_w = max(0, min(1, tw / crop_w))
                        norm_h = max(0, min(1, th / crop_h))
                        
                        # Map DENTEX class to 0-7 (Teeth 1 through 8)
                        # Note: Check the JSON; if category_ids are 0-31, map them to 0-7 using modulo
                        # CORRECTED DENTEX MAPPING:
                        # Grab the tooth number from the secondary ID key
                        tooth_id = int(ann['category_id_2']) 
                        
                        # YOLO requires 0-indexed classes. 
                        # If the tooth is 1-8, we subtract 1 to get 0-7. 
                        # If it is already 0-7, we leave it as is.
                        yolo_tooth_class = (tooth_id - 1) if tooth_id > 0 else 0
                        
                        f_label.write(f"{yolo_tooth_class} {norm_x} {norm_y} {norm_w} {norm_h}\n")

In [9]:
import torch
import gc
from ultralytics import YOLO

for q in range(1, 5):
    print(f"--- STARTING TRAINING FOR QUADRANT {q} ---")
    
    # 1. Create the YAML dynamically for this quadrant
    yaml_content = f"""
train: /kaggle/working/enumerators_data/Q{q}/images/train
val: /kaggle/working/enumerators_data/Q{q}/images/train 

nc: 8
names: ['Tooth_1', 'Tooth_2', 'Tooth_3', 'Tooth_4', 'Tooth_5', 'Tooth_6', 'Tooth_7', 'Tooth_8']
"""
    yaml_path = f'/kaggle/working/q{q}_data.yaml'
    with open(yaml_path, 'w') as f:
        f.write(yaml_content)

    # 2. Initialize a fresh lightweight YOLOv8 model
    model = YOLO('yolov8n.pt') 

    # 3. Train the enumerator model
    results = model.train(
        data=yaml_path,
        epochs=50,
        imgsz=416, # Reduced from 640. Because you cropped the X-rays, 416 is plenty large enough!
        batch=32,
        device=[0, 1], # Utilize both T4 GPUs
        project='/kaggle/working/runs',
        name=f'enumerator_q{q}'
    )
    
    # 4. CRITICAL: Clear Kaggle GPU memory before the next model starts
    del model
    del results
    gc.collect()
    torch.cuda.empty_cache()
    
    print(f"--- FINISHED QUADRANT {q}. GPU MEMORY CLEARED. ---\n")

--- STARTING TRAINING FOR QUADRANT 1 ---
Ultralytics 8.4.164 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/q1_data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, m

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/50     0.932G      1.617      2.525      1.547        200        416: 100% ━━━━━━━━━━━━ 20/20 7.5it/s 2.7s0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 6.6it/s 1.5s.2s
                   all        634       4607       0.57      0.154      0.356      0.143

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/50     0.932G      1.566      1.803      1.567        196        416: 100% ━━━━━━━━━━━━ 20/20 7.0it/s 2.8s0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 6.2it/s 1.6s.2s
                   all        634       4607       0.54      0.463      0.431      0.223

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/50     0.932G      1.542      1.565      1.561        203        416: 100% ━

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/50     0.932G      1.621      2.526      1.568        200        416: 100% ━━━━━━━━━━━━ 20/20 7.3it/s 2.7s0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 5.6it/s 1.8s.2s
                   all        634       4625      0.516       0.26      0.273      0.118

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/50     0.932G      1.602      1.891      1.605        210        416: 100% ━━━━━━━━━━━━ 20/20 7.1it/s 2.8s0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 6.2it/s 1.6s.2s
                   all        634       4625      0.537      0.435      0.474      0.218

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/50     0.932G      1.552      1.627      1.577        193        416: 100% ━

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/50     0.979G      1.571      2.385      1.417        202        416: 100% ━━━━━━━━━━━━ 20/20 6.8it/s 2.9s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 5.6it/s 1.8s.2s
                   all        634       5043      0.601      0.343      0.427      0.169

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/50     0.979G      1.521       1.72      1.439        217        416: 100% ━━━━━━━━━━━━ 20/20 7.1it/s 2.8s0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 6.1it/s 1.6s.2s
                   all        634       5043      0.737      0.503      0.696      0.363

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/50     0.979G      1.486      1.472      1.432        197        416: 100% ━

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/50     0.979G      1.601      2.434      1.448        216        416: 100% ━━━━━━━━━━━━ 20/20 6.9it/s 2.9s0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 5.2it/s 1.9s.2s
                   all        634       5039      0.506      0.358        0.5      0.244

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/50     0.979G      1.555       1.76      1.477        225        416: 100% ━━━━━━━━━━━━ 20/20 7.0it/s 2.8s0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 5.5it/s 1.8s.2s
                   all        634       5039      0.577      0.679      0.619      0.321

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/50     0.979G      1.486      1.494      1.454        199        416: 100% ━

In [10]:
import json
import os
import shutil

# 1. Define paths matching the DENTEX directory structure
json_path = '/kaggle/input/datasets/truthisneverlinear/dentex-challenge-2023/training_data/training_data/quadrant-enumeration-disease/train_quadrant_enumeration_disease.json'
img_dir = '/kaggle/input/datasets/truthisneverlinear/dentex-challenge-2023/training_data/training_data/quadrant-enumeration-disease/xrays/'
output_dir = '/kaggle/working/disease_data/'

os.makedirs(output_dir + 'images/train', exist_ok=True)
os.makedirs(output_dir + 'labels/train', exist_ok=True)

# 2. Load the JSON
with open(json_path, 'r') as f:
    coco_data = json.load(f)

# 3. Create a dynamic mapping for Disease IDs
# We must map DENTEX disease IDs to a continuous 0-3 range for YOLO
images = {img['id']: img for img in coco_data['images']}

unique_diseases = set()
for ann in coco_data['annotations']:
    if 'category_id_3' in ann:
        unique_diseases.add(int(ann['category_id_3']))

# This creates a dictionary mapping original IDs to [0, 1, 2, 3]
disease_mapping = {old_id: new_id for new_id, old_id in enumerate(sorted(list(unique_diseases)))}
print(f"Mapped Disease IDs {sorted(list(unique_diseases))} to YOLO classes {list(disease_mapping.values())}")

# 4. Generate YOLO Labels and isolate diseased images
annotated_img_ids = set()
for ann in coco_data['annotations']:
    # Skip teeth that are healthy (no category_id_3)
    if 'category_id_3' not in ann:
        continue
        
    img_info = images[ann['image_id']]
    img_width, img_height = img_info['width'], img_info['height']
    annotated_img_ids.add(ann['image_id'])
    
    # Standard COCO to YOLO normalized coordinate conversion
    x_min, y_min, w, h = ann['bbox']
    x_center = (x_min + w / 2) / img_width
    y_center = (y_min + h / 2) / img_height
    w_norm = w / img_width
    h_norm = h / img_height
    
    yolo_class_id = disease_mapping[int(ann['category_id_3'])]
    
    label_path = os.path.join(output_dir, 'labels/train', img_info['file_name'].replace('.jpg', '.txt').replace('.png', '.txt'))
    with open(label_path, 'a') as label_file:
        label_file.write(f"{yolo_class_id} {x_center} {y_center} {w_norm} {h_norm}\n")

# Copy the corresponding images to the YOLO training folder
for img_id in annotated_img_ids:
    img_name = images[img_id]['file_name']
    src = os.path.join(img_dir, img_name)
    dst = os.path.join(output_dir, 'images/train', img_name)
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy(src, dst)
        
print("Disease dataset successfully converted to YOLO format!")

Mapped Disease IDs [0, 1, 2, 3] to YOLO classes [0, 1, 2, 3]
Disease dataset successfully converted to YOLO format!


In [11]:
from ultralytics import YOLO

# Dynamically generate the data.yaml based on the mapped classes
class_names = [f"'Disease_{i}'" for i in range(len(disease_mapping))]
names_str = ", ".join(class_names)

yaml_content = f"""
train: {output_dir}images/train
val: {output_dir}images/train

nc: {len(disease_mapping)}
names: [{names_str}]
"""

with open('/kaggle/working/disease_data.yaml', 'w') as f:
    f.write(yaml_content)

# Initialize YOLOv8 Small for better feature extraction on subtle lesions
model = YOLO('yolov8s.pt') 

# Train using both T4 GPUs
results = model.train(
    data='/kaggle/working/disease_data.yaml',
    epochs=50,
    imgsz=640, 
    batch=32,
    device=[0, 1], 
    project='/kaggle/working/runs',
    name='disease_detector'
)

Ultralytics 8.4.164 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/disease_data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mos

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/50      3.71G      1.284       1.87      1.159         50        640: 100% ━━━━━━━━━━━━ 22/22 1.2it/s 18.2s0.4s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 11/11 1.8s/it 20.1s0.5s
                   all        678       3528      0.229      0.257      0.218      0.131

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/50      3.73G      1.212      1.736      1.143         18        640: 100% ━━━━━━━━━━━━ 22/22 1.1it/s 19.4s0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 11/11 1.8s/it 20.2s0.5s
                   all        678       3528      0.312      0.506      0.313      0.187

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/50      3.73G      1.262      1.719      1.179         21        640: 100% ━━━━━━━━━━━━ 22/22 1.1it/s 19.2s0.4s
                 Class     Images  In

In [13]:
import os
import cv2
import torch
from ultralytics import YOLO

# 1. Load All 6 Models
coordinator = YOLO('/kaggle/working/runs/coordinator_quadrant/weights/best.pt')
enumerators = {
    1: YOLO('/kaggle/working/runs/enumerator_q1/weights/best.pt'),
    2: YOLO('/kaggle/working/runs/enumerator_q2/weights/best.pt'),
    3: YOLO('/kaggle/working/runs/enumerator_q3/weights/best.pt'),
    4: YOLO('/kaggle/working/runs/enumerator_q4/weights/best.pt')
}
disease_detector = YOLO('/kaggle/working/runs/disease_detector/weights/best.pt')

# Map YOLO classes back to DENTEX diagnoses 
DISEASE_MAP = {
    0: 'Impacted', 
    1: 'Caries', 
    2: 'Periapical Lesion', 
    3: 'Deep Caries'
}

def calculate_iou(boxA, boxB):
    # Calculate Intersection over Union (IoU) between two bounding boxes [x1, y1, x2, y2]
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    interArea = max(0, xB - xA) * max(0, yB - yA)
    if interArea == 0: return 0.0

    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    return interArea / float(boxAArea + boxBArea - interArea)

def predict_full_pipeline(img_path):
    img = cv2.imread(img_path)
    if img is None:
        raise FileNotFoundError(f"Could not load image at {img_path}. Verify the path.")
    
    # A. Detect Diseases globally
    disease_results = disease_detector(img, verbose=False)[0].boxes
    diseases = []
    for d_box in disease_results:
        dx1, dy1, dx2, dy2 = map(int, d_box.xyxy[0].tolist())
        d_cls = int(d_box.cls[0].item())
        diseases.append({'box': [dx1, dy1, dx2, dy2], 'class': DISEASE_MAP.get(d_cls, f"Disease_{d_cls}")})
        
    # B. Detect Quadrants
    quadrant_results = coordinator(img, verbose=False)[0].boxes
    
    all_teeth = []
    
    # C. Detect Teeth per Quadrant
    for q_box in quadrant_results:
        q_cls = int(q_box.cls[0].item())
        q_num = q_cls + 1
        qx1, qy1, qx2, qy2 = map(int, q_box.xyxy[0].tolist())
        
        # Crop and run enumerator
        q_crop = img[qy1:qy2, qx1:qx2]
        if q_crop.shape[0] == 0 or q_crop.shape[1] == 0: continue
            
        tooth_results = enumerators[q_num](q_crop, verbose=False)[0].boxes
        
        for t_box in tooth_results:
            t_cls = int(t_box.cls[0].item())
            tooth_num = t_cls + 1 # YOLO classes 0-7 become Teeth 1-8
            
            # Shift local crop coordinates back to global full-image coordinates
            lx1, ly1, lx2, ly2 = map(int, t_box.xyxy[0].tolist())
            gx1, gy1, gx2, gy2 = lx1 + qx1, ly1 + qy1, lx2 + qx1, ly2 + qy1
            
            all_teeth.append({
                'quadrant': q_num,
                'tooth': tooth_num,
                'box': [gx1, gy1, gx2, gy2]
            })
            
    # D. Match Diseases to Teeth via IoU
    final_diagnoses = []
    for disease in diseases:
        best_iou = 0
        assigned_tooth = None
        
        for tooth in all_teeth:
            iou = calculate_iou(disease['box'], tooth['box'])
            if iou > best_iou:
                best_iou = iou
                assigned_tooth = tooth
                
        # Use a minimum overlap threshold to prevent false assignments (10% overlap minimum)
        if assigned_tooth and best_iou > 0.1: 
            final_diagnoses.append({
                'quadrant': assigned_tooth['quadrant'],
                'tooth': assigned_tooth['tooth'],
                'disease': disease['class']
            })
            
    return final_diagnoses

# 2. Dynamically locate an image to test
# Adjust this base path if your Kaggle directory structure is slightly different
xray_dir = '/kaggle/input/datasets/truthisneverlinear/dentex-challenge-2023/training_data/training_data/quadrant-enumeration-disease/xrays/'

if not os.path.exists(xray_dir):
    raise FileNotFoundError(f"Directory not found: {xray_dir}. Please check your Kaggle dataset paths.")

available_images = [f for f in os.listdir(xray_dir) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]

if not available_images:
    raise FileNotFoundError(f"No image files found in {xray_dir}.")

# Pick the first valid image found in the folder
test_img_path = os.path.join(xray_dir, available_images[0])
print(f"Testing on image: {test_img_path}\n")

# 3. Run the pipeline
predictions = predict_full_pipeline(test_img_path)

if not predictions:
    print("No abnormalities matched with teeth for this image.")
else:
    print("--- PIPELINE PREDICTIONS ---")
    for p in predictions:
        print(f"Quadrant: {p['quadrant']} | Tooth: {p['tooth']} | Diagnosis: {p['disease']}")

Testing on image: /kaggle/input/datasets/truthisneverlinear/dentex-challenge-2023/training_data/training_data/quadrant-enumeration-disease/xrays/train_429.png

--- PIPELINE PREDICTIONS ---
Quadrant: 3 | Tooth: 7 | Diagnosis: Impacted
Quadrant: 3 | Tooth: 5 | Diagnosis: Caries
Quadrant: 4 | Tooth: 7 | Diagnosis: Impacted
Quadrant: 2 | Tooth: 6 | Diagnosis: Caries
Quadrant: 1 | Tooth: 3 | Diagnosis: Caries


In [14]:
import os
from tqdm import tqdm

# Corrected path matching your Kaggle directory structure
val_img_dir = '/kaggle/input/datasets/truthisneverlinear/dentex-challenge-2023/validation_data/validation_data/quadrant_enumeration_disease/xrays/'

# Retrieve all image files in the validation directory
val_images = [f for f in os.listdir(val_img_dir) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]

print(f"Found {len(val_images)} unlabeled validation images. Running inference on a sample...")

# Loop through the first 10 images to inspect the results
for img_name in val_images[:10]:
    img_path = os.path.join(val_img_dir, img_name)
    print(f"\n--- {img_name} ---")
    
    try:
        preds = predict_full_pipeline(img_path)
        if not preds:
            print("No abnormalities matched with teeth.")
        else:
            for p in preds:
                print(f"Quadrant: {p['quadrant']} | Tooth: {p['tooth']} | Diagnosis: {p['disease']}")
    except Exception as e:
        print(f"Failed to process {img_name}. Error: {e}")

Found 50 unlabeled validation images. Running inference on a sample...

--- val_8.png ---
Quadrant: 4 | Tooth: 7 | Diagnosis: Deep Caries
Quadrant: 2 | Tooth: 5 | Diagnosis: Caries
Quadrant: 3 | Tooth: 5 | Diagnosis: Caries
Quadrant: 3 | Tooth: 6 | Diagnosis: Caries

--- val_6.png ---
Quadrant: 3 | Tooth: 7 | Diagnosis: Impacted
Quadrant: 4 | Tooth: 7 | Diagnosis: Impacted
Quadrant: 1 | Tooth: 7 | Diagnosis: Impacted

--- val_31.png ---
Quadrant: 3 | Tooth: 7 | Diagnosis: Impacted
Quadrant: 1 | Tooth: 7 | Diagnosis: Impacted
Quadrant: 4 | Tooth: 7 | Diagnosis: Impacted
Quadrant: 2 | Tooth: 7 | Diagnosis: Impacted
Quadrant: 1 | Tooth: 5 | Diagnosis: Caries
Quadrant: 2 | Tooth: 5 | Diagnosis: Caries
Quadrant: 4 | Tooth: 5 | Diagnosis: Caries
Quadrant: 4 | Tooth: 6 | Diagnosis: Caries
Quadrant: 3 | Tooth: 5 | Diagnosis: Caries

--- val_22.png ---
Quadrant: 3 | Tooth: 7 | Diagnosis: Caries
Quadrant: 2 | Tooth: 5 | Diagnosis: Caries
Quadrant: 4 | Tooth: 6 | Diagnosis: Caries
Quadrant: 3 | T

In [15]:
import shutil

# This will compress the entire 'runs' folder into a single file named 'dentex_pipeline_models.zip'
shutil.make_archive('/kaggle/working/dentex_pipeline_models', 'zip', '/kaggle/working/runs')

print("All models successfully zipped and ready for download!")

All models successfully zipped and ready for download!
